# 工具审批模式

In [1]:
from typing import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.types import Command, interrupt
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model("deepseek-flash")


# 两个工具：让模型一轮发起多个 tool_call，才能看出逐个审批的效果
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = {t.name: t for t in [get_weather, get_news]}
model_with_tools = model.bind_tools(tools.values())


# 声明状态
class State(MessagesState):
    decisions: dict  # {tool_call_id: 是否同意}


# 声明节点
def llm_node(state: State) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


def review_node(state: State) -> Command[Literal["tool_node", "llm_node", END]]:
    tool_calls = state["messages"][-1].tool_calls
    if not tool_calls:  # 模型直接回答，没有工具调用
        return Command(goto=END)
    # 工具执行前中断，列出全部待审批调用；恢复值是 {tool_call_id: 是否同意}
    decisions = interrupt([
        {
            "id": tc["id"],
            "name": tc["name"],
            "args": tc["args"]
        }
        for tc in tool_calls
    ])

    rprint(decisions)

    if any(decisions.values()):
        return Command(goto="tool_node", update={"decisions": decisions})
    # 全部拒绝 -> 回填拒绝 ToolMessage，回到 llm_node 让模型直接回答
    return Command(goto="llm_node", update={"messages": [
        ToolMessage(content=f"用户拒绝执行 {tc['name']}，请勿重试，直接回答", tool_call_id=tc["id"])
        for tc in tool_calls
    ]})


def tool_node(state: State) -> dict:
    last = state["messages"][-1]
    decisions = state["decisions"]
    # 同意的调用真正执行；拒绝的回填拒绝 ToolMessage，保证每个 tool_call_id 都有应答
    messages = []
    for tc in last.tool_calls:
        if decisions.get(tc["id"]):
            result = str(tools[tc["name"]].invoke(tc["args"]))
        else:
            result = f"用户拒绝执行 {tc['name']}，请勿重试，直接回答"
        messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    return {"messages": messages}


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("review_node", review_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "review_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("review_node", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断：等待用户对每个 tool_call 逐个审批
# 若模型不支持并行工具调用而分多轮发起，每轮各触发一次中断，恢复循环同样处理
res = graph.invoke({"messages": [HumanMessage("查一下北京的天气，再查一下科技新闻")]}, config=config)

rprint(res)

{
    'messages': [
        HumanMessage(
            content='查一下北京的天气，再查一下科技新闻',
            additional_kwargs={},
            response_metadata={},
            id='591b6c8c-08e8-488b-9def-74e923e8fb72'
        ),
        AIMessage(
            content='我来帮您同时查询北京天气和科技新闻。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user wants Beijing weather and tech news. Both are independent calls, so 
I can make them in the same block.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 102,
                    'prompt_tokens': 370,
                    'total_tokens': 472,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 24,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 242
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '50576095-716b-44b2-9ac8-75bce42e584e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0aa15-00a5-7281-8774-f93663ec3d7f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_Co3ZSqzQ6cC3ikZsETZv0835',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_4RBJuFVV3u0ZxeNxztYK4378',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 370,
                'output_tokens': 102,
                'total_tokens': 472,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {'reasoning': 24}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value=[
                {'id': 'call_00_Co3ZSqzQ6cC3ikZsETZv0835', 'name': 'get_weather', 'args': {'city': '北京'}},
                {'id': 'call_01_4RBJuFVV3u0ZxeNxztYK4378', 'name': 'get_news', 'args': {'topic': '科技'}}
            ],
            id='5a72296e51ea71db83e2455e2bc225f4'
        )
    ]
}

In [2]:
# from rich import print as rprint
#
# # 逐个审批 -> 恢复执行：对中断里的每个调用输入 y/n，按 tool_call_id 组装决策字典
# while res.get("__interrupt__"):
#     decisions = {}
#     for call in res["__interrupt__"][0].value:
#         user_in = input(f"是否允许 {call['name']}({call['args']})? (y/n)")
#         decisions[call["id"]] = user_in.strip().lower() in ("y", "yes", "是", "1")
#     res = graph.invoke(Command(resume=decisions), config=config)
#
# rprint(res)
#

{'call_00_Co3ZSqzQ6cC3ikZsETZv0835': True, 'call_01_4RBJuFVV3u0ZxeNxztYK4378': True}

{
    'messages': [
        HumanMessage(
            content='查一下北京的天气，再查一下科技新闻',
            additional_kwargs={},
            response_metadata={},
            id='591b6c8c-08e8-488b-9def-74e923e8fb72'
        ),
        AIMessage(
            content='我来帮您同时查询北京天气和科技新闻。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user wants Beijing weather and tech news. Both are independent calls, so 
I can make them in the same block.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 102,
                    'prompt_tokens': 370,
                    'total_tokens': 472,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 24,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 242
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '50576095-716b-44b2-9ac8-75bce42e584e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0aa15-00a5-7281-8774-f93663ec3d7f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_Co3ZSqzQ6cC3ikZsETZv0835',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_4RBJuFVV3u0ZxeNxztYK4378',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 370,
                'output_tokens': 102,
                'total_tokens': 472,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {'reasoning': 24}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            id='4a99e497-6795-4d8b-a7a5-a1c8b032e594',
            tool_call_id='call_00_Co3ZSqzQ6cC3ikZsETZv0835'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            id='3a71bc0c-84be-431c-91d3-910ffb5b38cc',
            tool_call_id='call_01_4RBJuFVV3u0ZxeNxztYK4378'
        ),
        AIMessage(
            content='查询结果如下：\n\n**🌤️ 北京天气**\n- 天气：晴天\n- 温度：25°C\n\n**📰 科技新闻**\n- 
最新科技新闻：AI 技术正在快速发展。\n\n北京今天天气晴朗、气温舒适（25°C），适合外出活动。科技方面，当前热点是 AI 
技术的快速发展。如需了解其他城市天气或其他主题（体育、娱乐）的新闻，随时告诉我！',
            additional_kwargs={'refusal': None, 'reasoning_content': ''},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 91,
                    'prompt_tokens': 513,
                    'total_tokens': 604,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
          

In [ ]:
# 打印图结构
from IPython.display import display

display(graph)